# Combinaisons de message functions — Expérimentation post-Approches 1-3

Notebook ablation : la composition `[message function locale + GlobalAggregation]` dans un `RecurrentCoupler` apporte-t-elle un bénéfice par rapport à la message function locale seule ?

Deux compositions sont testées :
- `LocalSum + GlobalAggregation`
- `GATv2 + GlobalAggregation`

Cette expérimentation a également servi d'étape empirique intermédiaire pour informer la conception du `VirtualAddressRecurrentCoupler` (Item 5, désormais livré). Le coupler de l'Item 5 introduit un état virtuel séparé `h_virtual_address`, ce qui est mécaniquement différent de la composition naïve testée ici.

**Prérequis :** avoir parcouru les notebooks `00_baseline_localsum`, `01_gatv2`, `02_global_aggregation` au préalable. Ce notebook reprend le même pipeline (Normalizer, Encoder, RecurrentCoupler, Decoder) et le même protocole expérimental.

**Durée attendue :** environ 30 à 40 minutes sur le serveur GPU de La Javaness (CUDA, JAX 0.9.0).


## 1. Mise en place de l'environnement

Imports : JAX, NumPy, Flax NNX, optax. Vérification du device JAX et du dtype par défaut.

In [1]:
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

# Repo root detection (works regardless of nbconvert cwd)
_root = Path.cwd().resolve()
while not (_root / "benchmarks").exists() and _root != _root.parent:
    _root = _root.parent
print(f"Working dir: {_root}")
print(f"JAX devices:    {jax.devices()}")
print(f"JAX default dtype: {jnp.zeros(1).dtype}")
print(f"NumPy version:  {np.__version__}")

Working dir: /home/van.tuan/GNN/energnn_attention


JAX devices:    [CudaDevice(id=0), CudaDevice(id=1)]


JAX default dtype: float32
NumPy version:  2.3.4


## 2. Méthode — Composition via `RecurrentCoupler.message_functions = list`

Le `RecurrentCoupler` du framework EnerGNN accepte une **liste** de message functions. À chaque pas Euler ($n_{steps}$ pas, $\delta t = 1/n_{steps}$), toutes les message functions sont appelées en parallèle, leurs sorties sont concaténées le long de l'axe feature, puis l'MLP $\phi$ réduit la concaténation à `latent_dim`.

Pour deux message functions de sortie `latent_dim` chacune :

$$h_a(t+1) = h_a(t) + \delta t \cdot \phi\!\left(\text{concat}\bigl(\psi^{\text{local}}(h, x)_a,\; \psi^{\text{global}}(h, x)_a\bigr)\right)$$

Conséquence sur la structure du modèle :
- `phi.in_size = 2 * latent_dim` (au lieu de `latent_dim` quand une seule message function)
- `phi.out_size = latent_dim` (inchangé)
- Le reste du pipeline (`Normalizer`, `Encoder`, `Decoder`) est inchangé

Cette composition n'est **pas** le `VirtualAddressRecurrentCoupler` (Item 5). L'Item 5 introduit un état virtuel séparé `h_virtual_address` qui évolue en parallèle de `h` via un `F_virtual` dédié. La composition naïve testée ici utilise uniquement le mécanisme de liste déjà disponible dans le `RecurrentCoupler` actuel, sans état virtuel.


# Partie 1 — Expériences sur LinearSystem

Composition vs message function locale seule sur le toy DC power flow. Le substrat (3 à 4 classes d'hyper-arêtes, quelques dizaines d'arêtes) sert de sanity check rapide ; son pouvoir discriminant entre mécanismes est faible (observé aux Approches 1-3).


## 3.1. Construction de `LinearSystemProblemLoader` et helper

Loader identique aux Approches précédentes (mêmes seeds, même dataset config). Le helper `build_composition_gnn` construit un GNN avec une liste de deux message functions à passer au `RecurrentCoupler`.

In [2]:
from energnn.model import (
    GNN, MLP, MLPEncoder, MLPEquivariantDecoder, RecurrentCoupler,
    GATv2MessagePassingFunction,
    GlobalAggregationMessagePassingFunction,
    LocalSumMessagePassingFunction,
    TDigestNormalizer,
)
from energnn.problem.example import LinearSystemProblemLoader
from energnn.trainer import Trainer

LATENT_DIM = 4  # Tiny config

ls_train = LinearSystemProblemLoader(seed=0)
ls_val = LinearSystemProblemLoader(seed=42)


def _make_localsum(in_struct, rngs):
    return LocalSumMessagePassingFunction(
        in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
        activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
        final_activation=None, outer_activation=nnx.tanh,
        encoded_feature_size=LATENT_DIM, rngs=rngs,
    )


def _make_gatv2(in_struct, rngs):
    return GATv2MessagePassingFunction(
        in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
        activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
        final_activation=None, outer_activation=nnx.tanh,
        encoded_feature_size=LATENT_DIM, rngs=rngs,
    )


def _make_globalagg(in_struct, rngs):
    return GlobalAggregationMessagePassingFunction(
        in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
        activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
        final_activation=None, outer_activation=nnx.tanh, rngs=rngs,
    )


def build_gnn(in_struct, out_struct, *, message_functions, rngs):
    """Build a GNN with the given list of message functions.

    `message_functions` is a list. If single, `phi.in_size = latent_dim`.
    If two, `phi.in_size = 2 * latent_dim`. Other parts unchanged.
    """
    n_messages = len(message_functions)
    return GNN(
        normalizer=TDigestNormalizer(in_structure=in_struct, n_breakpoints=10, update_limit=1000),
        encoder=MLPEncoder(
            in_structure=in_struct, hidden_sizes=[], activation=nnx.leaky_relu,
            out_size=LATENT_DIM, use_bias=True, final_activation=None, rngs=rngs,
        ),
        coupler=RecurrentCoupler(
            phi=MLP(
                in_size=n_messages * LATENT_DIM, hidden_sizes=[],
                activation=nnx.leaky_relu, out_size=LATENT_DIM,
                use_bias=True, final_activation=nnx.tanh, rngs=rngs,
            ),
            message_functions=message_functions,
            n_steps=5,
        ),
        decoder=MLPEquivariantDecoder(
            in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
            activation=nnx.leaky_relu, out_structure=out_struct, use_bias=True,
            final_activation=None, encoded_feature_size=LATENT_DIM, rngs=rngs,
        ),
    )


def count_params(model):
    _, params, _ = nnx.split(model, nnx.Param, ...)
    return sum(int(np.prod(p.shape)) for p in jax.tree_util.tree_leaves(params))


print(f"train loader: {type(ls_train).__name__}, seed=0")
print(f"val loader:   {type(ls_val).__name__}, seed=42")
print(f"build_gnn(...) helper ready (accepts a list of message_functions)")

train loader: LinearSystemProblemLoader, seed=0
val loader:   LinearSystemProblemLoader, seed=42
build_gnn(...) helper ready (accepts a list of message_functions)


## 3.2. Vérification de la structure de la composition

Sanity check : construire une composition `[LocalSum, GlobalAggregation]`, vérifier que `phi.in_size = 2 * LATENT_DIM`, faire un forward sur une instance et vérifier la forme de la sortie.

In [3]:
rngs_test = nnx.Rngs(0)
in_struct = ls_train.context_structure
out_struct = ls_train.decision_structure

# Construire la composition LocalSum + GlobalAggregation
msg_list = [_make_localsum(in_struct, rngs_test), _make_globalagg(in_struct, rngs_test)]
gnn_combo = build_gnn(in_struct, out_struct, message_functions=msg_list, rngs=rngs_test)

# Vérifier la structure du phi
phi_first_layer = gnn_combo.coupler.phi.sequential.layers[0]
print(f"phi first layer kernel shape: {phi_first_layer.kernel.value.shape}")
print(f"  attendu: ({2*LATENT_DIM}, {LATENT_DIM}) pour 2 message functions de sortie {LATENT_DIM}")

# Comptage des paramètres en composition vs standalone
n_params_combo = count_params(gnn_combo)
rngs_solo = nnx.Rngs(1)
gnn_solo = build_gnn(in_struct, out_struct,
    message_functions=[_make_localsum(in_struct, rngs_solo)], rngs=rngs_solo)
n_params_localsum = count_params(gnn_solo)
print(f"\nn_params composition (LocalSum + GlobalAgg): {n_params_combo}")
print(f"n_params LocalSum standalone:                {n_params_localsum}")
print(f"  surcoût composition : +{n_params_combo - n_params_localsum} parameters")

phi first layer kernel shape: (8, 4)
  attendu: (8, 4) pour 2 message functions de sortie 4

n_params composition (LocalSum + GlobalAgg): 221
n_params LocalSum standalone:                185
  surcoût composition : +36 parameters


/tmp/ipykernel_1700541/3337080139.py:11: DeprecationWarning: '.value' access is now deprecated. For Variable[Array] instances use:

  variable[...]

For other Variable types use:

  variable.get_value()

  print(f"phi first layer kernel shape: {phi_first_layer.kernel.value.shape}")


## 3.3. Entraînement sur LinearSystem (5 epochs par mécanisme)

Quatre GNN au pipeline identique à l'exception de la message function :
- `LocalSum` seul (référence)
- `GATv2` seul (référence)
- `LocalSum + GlobalAggregation` (composition 1)
- `GATv2 + GlobalAggregation` (composition 2)

Configuration : Tiny (`latent_dim=4`, `hidden_sizes=[]`, `n_steps=5`), 5 epochs, optimiseur `optax.adam(1e-3)`.


In [4]:
def train_and_eval(gnn, train_loader, val_loader, n_epochs):
    trainer = Trainer(model=gnn, gradient_transformation=optax.adam(1e-3))
    init, _ = trainer.eval(val_loader, progress_bar=False)
    curve = [float(init)]
    for _ in range(n_epochs):
        trainer.train(train_loader=train_loader, val_loader=None, n_epochs=1,
                      progress_bar=False, eval_before_training=False, eval_after_epoch=False)
        s, _ = trainer.eval(val_loader, progress_bar=False)
        curve.append(float(s))
    return curve


N_EPOCHS_LS = 5
ls_curves = {}
for label, msg_factory in (
    ("LocalSum",                lambda r: [_make_localsum(in_struct, r)]),
    ("GATv2",                   lambda r: [_make_gatv2(in_struct, r)]),
    ("LocalSum + GlobalAgg",    lambda r: [_make_localsum(in_struct, r), _make_globalagg(in_struct, r)]),
    ("GATv2 + GlobalAgg",       lambda r: [_make_gatv2(in_struct, r), _make_globalagg(in_struct, r)]),
):
    rngs = nnx.Rngs(0)
    gnn = build_gnn(in_struct, out_struct, message_functions=msg_factory(rngs), rngs=rngs)
    n_params = count_params(gnn)
    ls_curves[label] = train_and_eval(gnn, ls_train, ls_val, N_EPOCHS_LS)
    print(f"{label:<24s} n_params={n_params:>5d}  init={ls_curves[label][0]:.3e}  final={ls_curves[label][-1]:.3e}")

print(f"\n{'epoch':<8s} {'LocalSum':>14s} {'GATv2':>14s} {'LS+GA':>14s} {'GATv2+GA':>14s}")
print("-" * 80)
for ep in range(N_EPOCHS_LS + 1):
    label = "init" if ep == 0 else f"ep {ep}"
    vals = [ls_curves[k][ep] for k in ("LocalSum", "GATv2", "LocalSum + GlobalAgg", "GATv2 + GlobalAgg")]
    print(f"{label:<8s} {vals[0]:>14.4e} {vals[1]:>14.4e} {vals[2]:>14.4e} {vals[3]:>14.4e}")

LocalSum                 n_params=  185  init=5.021e-01  final=3.941e-01


GATv2                    n_params=  232  init=1.616e+00  final=1.455e+00


LocalSum + GlobalAgg     n_params=  221  init=7.916e-01  final=6.930e-01


GATv2 + GlobalAgg        n_params=  268  init=3.139e+00  final=2.805e+00

epoch          LocalSum          GATv2          LS+GA       GATv2+GA
--------------------------------------------------------------------------------
init         5.0206e-01     1.6156e+00     7.9165e-01     3.1395e+00
ep 1         4.7552e-01     1.5652e+00     7.6814e-01     3.0090e+00
ep 2         4.5123e-01     1.5371e+00     7.4889e-01     2.9544e+00
ep 3         4.2968e-01     1.5097e+00     7.2997e-01     2.9041e+00
ep 4         4.1095e-01     1.4820e+00     7.1133e-01     2.8535e+00
ep 5         3.9414e-01     1.4550e+00     6.9296e-01     2.8051e+00


# Partie 2 — Expériences sur IEEE-9 et IEEE-14 (AC load flow)

Substrat plus discriminant que LinearSystem : 11 classes d'hyper-arêtes, AC LF non linéaire, oracle = V_mag / P / Q / I obtenus par le solver pypowsybl. C'est sur ce substrat que les Approches 1-3 ont montré des différences claires entre mécanismes.

## 4.1. Construction de `ACLoadFlowProblemLoader` pour ieee9 et ieee14

Loader pre-cache (infrastructure partagée établie à l'Approche 1) : génère `dataset_size` instances une fois à l'`__init__` avec seed contrôlé, puis cache les paires `(context, oracle)`. Évite toute dérive du solver pypowsybl entre exécutions.

In [5]:
import time
sys.path.insert(0, str(_root / "benchmarks"))
from ac_load_flow_problem import ACLoadFlowProblemLoader

LOADERS = {}
for net in ("ieee9", "ieee14"):
    t0 = time.perf_counter()
    LOADERS[net] = {
        "train": ACLoadFlowProblemLoader(network_name=net, seed=7, perturbation_scale=0.1, dataset_size=16, batch_size=4),
        "val":   ACLoadFlowProblemLoader(network_name=net, seed=8, perturbation_scale=0.1, dataset_size=8,  batch_size=4),
    }
    print(f"  {net} loaders ready ({time.perf_counter() - t0:.1f}s)")

  ieee9 loaders ready (3.3s)


  ieee14 loaders ready (3.3s)


## 4.2. Entraînement sur ieee9 et ieee14 (3 epochs par mécanisme)

Run court (3 epochs Tiny) pour avoir une démonstration visuelle. Pour les chiffres canoniques (3 seeds, 10-15 epochs), voir la section 4.3 ci-dessous qui charge les résultats des runs complets.

In [6]:
N_EPOCHS_IEEE = 3
ieee_results = {}

for net in ("ieee9", "ieee14"):
    train_loader = LOADERS[net]["train"]
    val_loader = LOADERS[net]["val"]
    in_s = train_loader.context_structure
    out_s = train_loader.decision_structure
    ieee_results[net] = {}
    for label, msg_factory in (
        ("LocalSum",                lambda r, s: [_make_localsum(s, r)]),
        ("GATv2",                   lambda r, s: [_make_gatv2(s, r)]),
        ("LocalSum + GlobalAgg",    lambda r, s: [_make_localsum(s, r), _make_globalagg(s, r)]),
        ("GATv2 + GlobalAgg",       lambda r, s: [_make_gatv2(s, r), _make_globalagg(s, r)]),
    ):
        rngs = nnx.Rngs(0)
        gnn = build_gnn(in_s, out_s, message_functions=msg_factory(rngs, in_s), rngs=rngs)
        t0 = time.perf_counter()
        curve = train_and_eval(gnn, train_loader, val_loader, N_EPOCHS_IEEE)
        ieee_results[net][label] = {"curve": curve, "n_params": count_params(gnn),
                                     "train_time_s": time.perf_counter() - t0}

print(f"{'network':<8s} {'mechanism':<24s} {'n_params':>9s} {'init':>12s} {'final':>12s} {'time (s)':>10s}")
print("-" * 80)
for net in ("ieee9", "ieee14"):
    for label in ("LocalSum", "GATv2", "LocalSum + GlobalAgg", "GATv2 + GlobalAgg"):
        r = ieee_results[net][label]
        print(f"{net:<8s} {label:<24s} {r['n_params']:>9d} {r['curve'][0]:>12.3e} {r['curve'][-1]:>12.3e} {r['train_time_s']:>10.1f}")

network  mechanism                 n_params         init        final   time (s)
--------------------------------------------------------------------------------
ieee9    LocalSum                      1587    6.904e-01    5.922e-01       68.3
ieee9    GATv2                         1881    5.762e-01    5.250e-01       70.0
ieee9    LocalSum + GlobalAgg          1623    6.193e-01    5.071e-01       52.2
ieee9    GATv2 + GlobalAgg             1917    5.627e-01    5.159e-01       66.7
ieee14   LocalSum                      1587    3.823e-01    3.149e-01       86.6
ieee14   GATv2                         1881    3.974e-01    3.491e-01       93.4
ieee14   LocalSum + GlobalAgg          1623    3.299e-01    2.521e-01       69.5
ieee14   GATv2 + GlobalAgg             1917    3.618e-01    3.272e-01       86.2


## 4.3. Référence aux runs canoniques (3 seeds × Tiny/Small × 10-15 epochs)

Les résultats ci-dessus sont obtenus avec un budget court (3 epochs Tiny). Les chiffres canoniques sont disponibles dans `benchmarks/results/05_combinations/baseline_combinations_ac_load_flow.json` et dans les JSONs des Approches 0, 1, 2 pour référence.

In [7]:
import json
import statistics

REPO = _root  # we're at the repo root for runtime
RESULTS = REPO / "benchmarks" / "results"


def median_best_eval(runs, *, size, network=None):
    matches = [min(r["epoch_eval_curve"]) for r in runs
               if r["size"] == size and (network is None or r.get("network") == network)]
    return statistics.median(matches) if matches else None


baselines = {
    "LocalSum":   json.load((RESULTS / "00_baseline" / "baseline_ac_load_flow.json").open())["runs"],
    "GATv2":      json.load((RESULTS / "01_gatv2" / "baseline_gatv2_ac_load_flow.json").open())["runs"],
    "GlobalAgg":  json.load((RESULTS / "02_global_aggregation" / "baseline_global_aggregation_ac_load_flow.json").open())["runs"],
}

combos = json.load((RESULTS / "05_combinations" / "baseline_combinations_ac_load_flow.json").open())["runs"]
ls_ga = [r for r in combos if r["combo"] == "LocalSum+GlobalAgg"]
gv_ga = [r for r in combos if r["combo"] == "GATv2+GlobalAgg"]


print("Best-eval médian sur AC LF (3 seeds, Tiny/Small):\n")
print(f"{'Mécanisme':<28s} {'ieee9 Tiny':>14s} {'ieee9 Small':>14s} {'ieee14 Tiny':>14s} {'ieee14 Small':>14s}")
print("-" * 90)
for name, runs in [
    ("LocalSum (référence)", baselines["LocalSum"]),
    ("GATv2 standalone",     baselines["GATv2"]),
    ("GlobalAgg standalone", baselines["GlobalAgg"]),
    ("LocalSum + GlobalAgg", ls_ga),
    ("GATv2 + GlobalAgg",    gv_ga),
]:
    row = [median_best_eval(runs, size=s, network=n) for n, s in (("ieee9","Tiny"), ("ieee9","Small"), ("ieee14","Tiny"), ("ieee14","Small"))]
    cells = [f"{v:>14.3e}" if v else f"{'—':>14s}" for v in row]
    print(f"{name:<28s} {' '.join(cells)}")

Best-eval médian sur AC LF (3 seeds, Tiny/Small):

Mécanisme                        ieee9 Tiny    ieee9 Small    ieee14 Tiny   ieee14 Small
------------------------------------------------------------------------------------------
LocalSum (référence)              1.561e-01      5.075e-03      8.668e-02      4.564e-03
GATv2 standalone                  1.392e-01      5.837e-03      8.280e-02      7.995e-03
GlobalAgg standalone              2.469e-01      2.662e-02      1.482e-01      1.846e-02
LocalSum + GlobalAgg              1.958e-01      1.171e-02      1.241e-01      8.971e-03
GATv2 + GlobalAgg                 2.444e-01      9.568e-03      1.671e-01      8.995e-03


## 4.4. Test de l'hypothèse : composition vs locale seule

Lecture des résultats : pour chaque comparaison, la composition fait-elle mieux que la message function locale seule ?

In [8]:
def delta(alone, combo):
    if alone is None or combo is None:
        return "n/a"
    diff = (combo - alone) / alone * 100
    sign = "+" if diff > 0 else ""
    if abs(diff) < 5:
        verdict = "SAME"
    elif diff < 0:
        verdict = "BETTER"
    else:
        verdict = "WORSE"
    return f"{sign}{diff:.1f}%  ({verdict})"


configs = [("ieee9", "Tiny"), ("ieee9", "Small"), ("ieee14", "Tiny"), ("ieee14", "Small")]

print("LocalSum + GlobalAgg vs LocalSum standalone:")
for net, sz in configs:
    a = median_best_eval(baselines["LocalSum"], size=sz, network=net)
    c = median_best_eval(ls_ga, size=sz, network=net)
    print(f"  {net} {sz:<5s}: {delta(a, c)}")

print("\nGATv2 + GlobalAgg vs GATv2 standalone:")
for net, sz in configs:
    a = median_best_eval(baselines["GATv2"], size=sz, network=net)
    c = median_best_eval(gv_ga, size=sz, network=net)
    print(f"  {net} {sz:<5s}: {delta(a, c)}")

LocalSum + GlobalAgg vs LocalSum standalone:
  ieee9 Tiny : +25.4%  (WORSE)
  ieee9 Small: +130.8%  (WORSE)
  ieee14 Tiny : +43.2%  (WORSE)
  ieee14 Small: +96.6%  (WORSE)

GATv2 + GlobalAgg vs GATv2 standalone:
  ieee9 Tiny : +75.5%  (WORSE)
  ieee9 Small: +63.9%  (WORSE)
  ieee14 Tiny : +101.8%  (WORSE)
  ieee14 Small: +12.5%  (WORSE)


## 5. Synthèse et perspectives

Résumé empirique de l'expérimentation :

- **Sur LinearSystem** (substrat toy `n_max=3`) : les mécanismes sont peu discriminés. Le standalone `LocalSum` converge vers 0,39 ; les compositions et le standalone `GATv2` restent entre 0,69 et 2,81. Ce substrat reste peu informatif pour distinguer les mécanismes (observation déjà notée aux Approches 1-3).

- **Sur IEEE AC LF (ieee9, ieee14)** : la composition est **plus mauvaise** que la message function locale seule dans 8 configurations sur 8 testées (Tiny et Small × ieee9 et ieee14, pour les deux compositions). Les écarts vont de +12 % à +131 % de best-eval médiane par rapport au standalone.

- **Interprétations possibles** :
  1. Le mécanisme de composition naïf — concaténation des messages dans $\phi$ — peut diluer le signal local sans apporter de signal global utile aux échelles ieee9/14.
  2. Le `phi.in_size` est doublé (de `latent_dim` à `2 * latent_dim`) ; sa profondeur, calibrée pour l'entrée single-input, n'a pas été ajustée. Une ablation `phi` plus profond serait nécessaire pour départager cet axe.
  3. À cette échelle, les `n_steps` du `RecurrentCoupler` (5 à 10 pas Euler) propagent déjà l'information localement sur l'ensemble du graphe, rendant le contexte global de `GlobalAggregation` redondant.

- **Lien avec le `VirtualAddressRecurrentCoupler` (Item 5, livré)** : la composition naïve testée ici (concaténation dans $\phi$) diffère du coupler à état virtuel séparé (`h_virtual_address` évolué par `F_virtual` dédié). Le notebook a motivé empiriquement l'introduction d'un état virtuel séparé. La v1 livrée (VAR+LocalSum) ne dépasse pas non plus LocalSum seul à 3 seeds (cf. `BASELINES.md` Approche 5).

**Référence empirique** : `benchmarks/results/05_combinations/baseline_combinations_ac_load_flow.json` (24 runs canoniques : 2 compositions × 2 réseaux × 2 tailles × 3 seeds).
